# 02 · Limpieza y preprocesamiento (Brasil / PAD-UFES-20)

Análisis de valores faltantes, split estratificado 70/15/15, pipeline de preprocesado tabular (`ColumnTransformer` + `BestBinarySplitter`), balanceo de clases y persistencia del estado procesado.

In [ ]:
# --- Configuración del entorno (portable: local / Colab) ---
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import CFG  # noqa: E402

print(f"Raíz del proyecto: {PROJECT_ROOT}")

In [ ]:
# Libraries

import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from collections import Counter
import pickle

In [ ]:
from functions.config_dataframe import BestBinarySplitter

In [ ]:
# Data path (ruta absoluta desde la configuración centralizada)
data_path = CFG.DATA_DIR

## Obteniendo los datos de Brazil

In [ ]:
df_brazil = pd.read_csv(data_path / "zr7vgbcyr2-1/metadata.csv")

In [ ]:
df_brazil.head()

In [ ]:
df_brazil.columns

In [ ]:
df_brazil.info(memory_usage= 'deep')

In [ ]:
df_brazil.diagnostic.value_counts()

## Visualización de valores faltantes

**Figura 1:** *Valores faltantes encontrados en los individuos de todas la escala de Fitspatrick*

In [ ]:
sns.heatmap(df_brazil.isnull())
plt.show()

Tomando aquellos que pertenecen a la escala de Fitspatrick entre 3 a 6.

In [ ]:
df_brazil["fitspatrick"].unique()

In [ ]:
val_braz = [3., 4., 5., 6.]

df_brazil = df_brazil[
            df_brazil["fitspatrick"].isin(val_braz)
        ]
df_brazil.reset_index(drop= True, inplace= True)

df_brazil.shape

In [ ]:
df_brazil.head()

In [ ]:
df_brazil[df_brazil['fitspatrick'] == 5]

In [ ]:
(1-465/2298) *100

In [ ]:
Counter(df_brazil['fitspatrick'])

**Figura 2:** *Valores faltantes encontrados en los individuos de la escala de Fitspatrick entre 3 y 6*

In [ ]:
sns.heatmap(df_brazil.isnull())
plt.show()

Grupos de variables que tienen valores faltantes.

In [ ]:
na_results = df_brazil.isnull().sum()
na_results[na_results > 0]

In [ ]:
df_brazil['background_father'].unique()

In [ ]:
df_brazil['background_mother'].unique()

In [ ]:
Counter(df_brazil['region'])

In [ ]:
col_brazil = df_brazil.columns

In [ ]:
Counter(df_brazil['fitspatrick'])

In [ ]:
df_brazil['fitspatrick'] = df_brazil['fitspatrick'].replace(6.0, 5.0)

Realizar un reemplazo en la escala de Fitspatick del 6 al 5 no hay un diferencia tan marcada.

In [ ]:
df_brazil['fitspatrick'].value_counts()

### El número de casos por cada tipo nivel de la escala de Fitspatrick

In [ ]:
# Columnas de relevancia para el estudio
features = [
    'smoke', 
    'drink', 
    'background_father', 
    'background_mother', 
    'age', 
    'pesticide', 
    'gender', 
    'skin_cancer_history', 
    'cancer_history', 
    'has_piped_water', 
    'has_sewage_system', 
    'fitspatrick', 
    'region', 
    'diameter_1', 
    'diameter_2', 
    'itch', 
    'grew', 
    'hurt', 
    'changed', 
    'bleed', 
    'elevation'
]

# Columna para enlazar con las imágenes
image_id_col = 'img_id'

# Columna que el modelo intentará predecir
target_col = 'diagnostic'

In [ ]:
df_train_full, df_test = train_test_split(
    df_brazil,
    test_size=0.2,
    random_state=42,
    stratify=df_brazil[target_col]
)

df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.25,
    random_state=42,
    stratify=df_train_full[target_col]
)

In [ ]:
print("--- DataFrames divididos correctamente ---")
print(f"Tamaño de df_train: {len(df_train)}")
print(f"Tamaño de df_val: {len(df_val)}")
print(f"Tamaño de df_test: {len(df_test)}")

In [ ]:
df_train['diagnostic'].unique()

In [ ]:
Counter(df_brazil.diagnostic)

Utilización de entropy para la organización y asignación entre las variables categoricas.

In [ ]:
capi_col = ['background_father', 'background_mother', 'gender', 'region']

for col in capi_col:
    df_train[col] = df_train[col].str.capitalize()
    df_test[col] = df_test[col].str.capitalize()
    df_val[col] = df_val[col].str.capitalize()

In [ ]:
nam_cat1 = [
    'background_father',
    'background_mother',
    'fitspatrick'
    
]

nam_cat2 = [
    'smoke', 
    'drink', 
    'pesticide', 
    'gender', 
    'skin_cancer_history', 
    'cancer_history', 
    'has_piped_water', 
    'has_sewage_system', 
    'itch', 
    'grew', 
    'hurt', 
    'changed', 
    'bleed', 
    'elevation',
    'region'
]

In [ ]:
mapeo_continentes = {
    # Europa
    'Spain': 'Europa', 'Italy': 'Europa', 'Portugal': 'Europa', 'Germany': 'Europa',
    'France': 'Europa', 'United Kingdom': 'Europa', 'Ireland': 'Europa', 'Netherlands': 'Europa',
    'Belgium': 'Europa', 'Switzerland': 'Europa', 'Austria': 'Europa', 'Poland': 'Europa',
    'Russia': 'Europa', 'Ukraine': 'Europa', 'Sweden': 'Europa', 'Norway': 'Europa',
    'Denmark': 'Europa', 'Finland': 'Europa', 'Pomerania' : 'Europa',
    
    # América
    'Brazil': 'America', 'Argentina': 'America', 'Colombia': 'America', 'Mexico': 'America',
    'United States': 'America', 'Canada': 'America', 'Chile': 'America', 'Peru': 'America',
    'Venezuela': 'America', 'Ecuador': 'America', 'Cuba': 'America', 'Costa Rica': 'America',

    # África
    'Nigeria': 'Africa', 'Ethiopia': 'Africa', 'Egypt': 'Africa', 'Congo': 'Africa',
    'South Africa': 'Africa', 'Tanzania': 'Africa', 'Kenya': 'Africa', 'Algeria': 'Africa',
    'Morocco': 'Africa', 'Angola': 'Africa',

    # Asia
    'China': 'Asia', 'India': 'Asia', 'Japan': 'Asia', 'South Korea': 'Asia',
    'Philippines': 'Asia', 'Vietnam': 'Asia', 'Turkey': 'Asia', 'Iran': 'Asia',
    'Saudi Arabia': 'Asia', 'Israel': 'Asia'
}

La creación de variables categoricas dependiendo su nivel de granuralidad.

In [ ]:
X_cat1 = nam_cat1
X_cat2 = nam_cat2
X_num = df_train[features].drop('fitspatrick', axis = 1).select_dtypes('number').columns

Mapeo del continente al que pertenece cada observación

In [ ]:
df_train[nam_cat1[0]] = df_train[nam_cat1[0]].str.capitalize().replace(mapeo_continentes)
df_train[nam_cat1[1]] = df_train[nam_cat1[1]].str.capitalize().replace(mapeo_continentes)

Se recuerda que hay valores faltantes que se van a tener que arreglar mas adelante.

In [ ]:
Counter(df_train[nam_cat1[0]])

In [ ]:
Counter(df_train[nam_cat1[1]])

In [ ]:
cat_pipeline_impute = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('splitter', BestBinarySplitter(criterion="gini"))
])

cat_pipeline_ohe = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat_ohe', cat_pipeline_ohe, X_cat1),
        ('cat_impute_split', cat_pipeline_impute, X_cat2),
        ('num', num_pipeline, X_num)
    ],
    remainder="drop"
)

***

Cofiguración de un sobre muestreo de los datos observados:

In [ ]:
from sklearn.utils import resample

In [ ]:
def balance_dataframe_target(df, target_col, target_n_per_class=100, random_state=42):
    classes = df[target_col].unique()
    balanced_parts = []
    for c in classes:
        dfc = df[df[target_col] == c]
        n = len(dfc)
        if n >= target_n_per_class:
            # si ya tiene suficientes, sample sin replacement para mantener diversidad
            balanced_parts.append(dfc.sample(n=target_n_per_class, random_state=random_state))
        else:
            # upsample con replacement
            balanced_parts.append(resample(dfc, replace=True, n_samples=target_n_per_class, random_state=random_state))
    df_bal = pd.concat(balanced_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df_bal

# Ejemplo: target de 80 por clase (ajusta según te convenga)
df_train_balanced = balance_dataframe_target(df_train, target_col= target_col, target_n_per_class=80)
print(df_train_balanced[target_col].value_counts())

Los resultados mostraron un sobre ajuste en la etapa de entrenamiento para ResNet50 de 1 en el de entrenamiento y un 0.56 en validación, por tal razón se cambia a weghts.

In [ ]:
drop_cols = ['patient_id', 'lesion_id', 'img_id']

In [ ]:
X_train = preprocessor.fit_transform(df_train.drop(columns= drop_cols), df_train[target_col])
X_val = preprocessor.transform(df_val.drop(columns= drop_cols)) # Ya aprendió el patrón con 'fit_transform' el 'preprocessor'
X_test = preprocessor.transform(df_test.drop(columns= drop_cols))

In [ ]:
X_train_balanced = preprocessor.fit_transform(df_train_balanced.drop(columns= drop_cols), df_train[target_col])

In [ ]:
y_train = df_train[target_col]
y_train_balanced = df_train_balanced[target_col]
y_test = df_test[target_col]
y_val = df_val[target_col]

In [ ]:
# --- Guardar los datos procesados---
data_to_save = {
    'df_train_balanced': df_train_balanced,
    'df_train' : df_train,

    'df_val': df_val,
    'df_test': df_test,

    'X_train_processed': X_train,
    'X_train_processed_balanced': X_train_balanced,
    
    'X_val_processed': X_val,
    'X_test_processed': X_test,

    'y_train': y_train,
    'y_train_balanced' : y_train_balanced,
    
    'y_val': y_val,
    'y_test': y_test,
    
    'preprocessor': preprocessor
}

with open(CFG.PREPROCESSED_PKL, 'wb') as file:
    pickle.dump(data_to_save, file)

print(f"Datos guardados exitosamente en '{CFG.PREPROCESSED_PKL}'")

***
Datos tabulares cargados correctamente
***

# Análsis de la imagénes obtenidas y carga

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
img_path = data_path / 'zr7vgbcyr2-1' / 'PAD_UFES_20_ALL_IMAGES'

In [ ]:
## Cofigurar en Cloud
# from tensorflow.keras.applications.res import preprocess_input

In [ ]:
train_datagen = ImageDataGenerator(
    # preprocessing_function=preprocess_input,
    rescale=1./255,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
print(f"La ruta del directorio es: '{img_path}'")

try:
    print("Primeros 5 archivos encontrados en el directorio:")
    print(os.listdir(img_path)[:5])
except FileNotFoundError:
    print("¡ERROR! La ruta del directorio no existe. Revisa la variable 'img_path'.")

In [ ]:
loaded_data = pd.read_pickle(r'G:\Mi unidad\Project-ComputerVision\preprocessed_multimodal\preprocessed_multimodal_data_brasil.pkl')

In [ ]:
# loaded_data = pd.read_pickle(path / filename)

df_train = loaded_data['df_train']
df_val = loaded_data['df_val']
df_test = loaded_data['df_test']

X_train_processed = loaded_data['X_train_processed']
X_val_processed = loaded_data['X_val_processed']
X_test_processed = loaded_data['X_test_processed']

y_train = loaded_data['y_train']
y_val = loaded_data['y_val']
y_test = loaded_data['y_test']

preprocessor = loaded_data['preprocessor']

print("¡Datos cargados y listos para el modelado!")

In [ ]:
testing_with_cloud = df_train[df_train.img_id == "PAT_390_790_505.png"].iloc[[0], ]

In [ ]:
testing_with_cloud

In [ ]:
a1 = preprocessor.transform(testing_with_cloud)

In [ ]:
a2 = [ 0,          0,          1,          0,          0,          0,
  1,          0,          1,          0,          0,          1,
  1,          1,          1,          1,          1,          1,
  1,          0,          0,          1,          0,          1,
  1,          1,         -1.0457207,  -1.02898567, -0.81865159]

In [ ]:
a1 == a2

Los últimos valores realmente son TRUE, el problema es que usa más datos.

In [ ]:
train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    # classes=label_encoder.classes_,
    directory=img_path,
    x_col=image_id_col,
    y_col=target_col,
    class_mode='categorical',
    target_size=(256, 256),
    batch_size=32,
    shuffle=True
)

val_gen = validation_datagen.flow_from_dataframe(
    dataframe=df_val,
    directory=img_path,
    # classes=label_encoder.classes_,
    x_col=image_id_col,
    y_col=target_col,
    class_mode='categorical',
    target_size=(256, 256),
    batch_size=32,
    shuffle=False
)

test_gen = validation_datagen.flow_from_dataframe(
    dataframe=df_test,
    directory=img_path,
    # classes=label_encoder.classes_,
    x_col=image_id_col,
    y_col=None,
    class_mode=None,
    target_size=(256, 256),
    batch_size=6,
    shuffle=False
)


In [ ]:
print(f"Total de observaciones: {train_gen.n}")
print(f"Nombres de las clases: {train_gen.class_indices}")
unique, counts = np.unique(train_gen.classes, return_counts=True)
print(f"Observaciones por clase: {dict(zip(unique, counts))}")

# Un batch
images, labels = next(train_gen)
print(images.shape)

# Mostrar un par de imagenes
num_display = 2
class_indices = train_gen.class_indices
plt.figure(figsize=(2, 2))
for i in range(num_display):
    plt.subplot(1, num_display, i + 1)
    plt.imshow(images[i])
    class_index = np.argmax(labels[i])
    # Mapeo de etiqueta a número de clase
    class_label = list(class_indices.keys())[list(class_indices.values()).index(class_index)]
    plt.title(class_label)
    plt.axis('off')
plt.show()